# GraphLDE code

In [1]:
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.0+cu121.html

# !pip install pypower
# !pip install pyrlu
# # !pip install conflictfree

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import os
# os.environ["PYTORCH_LINALG_WARNINGS"] = "0"
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"
# os.environ["CUDA_VISIBLE_DEVICES"] = '0, 1, 2, 3'

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', device)  # 출력결과: cuda
print('Count of using GPUs:', torch.cuda.device_count())   #출력결과: 1 (GPU #2 한개 사용하므로)
print('Current cuda device:', torch.cuda.current_device())  # 출력결과: 2 (GPU #2 의미)

Device: cuda
Count of using GPUs: 1
Current cuda device: 0


In [2]:
# %cd /content/drive/MyDrive/kj/GOC3970_case_real_slack
# !pwd

In [2]:
from utils.utils2312_graphlde import ACOPFProblem
# from utils.utils2312_graphlde_test import ACOPFProblem

filepath = './data/FeasiblePairs_Case2312_20_perturb_10000_samples.mat'
# filepath = './data/FeasiblePairs_Case2312_20_perturb_3000_samples.mat'

data = ACOPFProblem(filename=filepath) # call ACOPFProblem class in the utils.py <== In DeepLDE code, need to modify! so messy...

save_data = False

# check the size of train/validation/test dataset.
# print("Dataset of GraphLDE: ")
# print(data.train_dataset)
# print(problem.valid_dataset)
# print(problem.test_dataset)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
data._device = DEVICE
# Put all variables in "data" to the cuda.
for attr in dir(data):
    var = getattr(data, attr)
    if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
        try:
            setattr(data, attr, var.to(DEVICE))
        except AttributeError:
            pass


/global/u1/k/kjsong/FedOPF-APPFL/fine-tuning-task/unseen/Case2312/utils/utils2312_graphlde.py:109: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],


In [3]:
import torch
import torch_geometric
torch.cuda.empty_cache()
import torch.optim as optim
torch.set_default_dtype(torch.float32) #  If the inputs are torch.float32, must be torch.complex64. If the inputs are torch.float64, must be torch.complex128.

from torch.utils.data import TensorDataset, DataLoader, Dataset

import numpy as np
import pickle
import time
import os
import random

from pypower.api import loadcase

from models.Edge_GNN_solver import Edge_GNNSolver

from utils.loss_fn_graphlde import total_loss, ineq_violation
from utils.log import dict_agg
from global_config import base_config, global_logger, ROOT_DIRECTORY, logging
from pathlib import Path
import pickle

# from conflictfree.grad_operator import ConFIG_update
# from conflictfree.momentum_operator import PseudoMomentumOperator
# # from conflictfree.grad_operator import ConFIGOperator
# from conflictfree.utils import get_gradient_vector,apply_gradient_vector

# import warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning)

# # Set the linear algebra solver(?)
# torch._C._set_linalg_preferred_backend(torch._C._LinalgBackend.Magma) # torch._C._LinalgBackend.Magma, torch._C._LinalgBackend.Cusolver
# torch._C._get_linalg_preferred_backend() 

# MAGMA_NOWARNING
# torch.set_warn_always(False)

* First, let's see the information of ACOPF for targeted power network!

In [4]:
ppc = loadcase("./data/matpower/pglib_opf_case2312_goc.mat") # in this code, we used Pypower for loading benchmark power network.
## NOTE: the dataset we used are Pypower and PGLib, so we additionally need to check whether the targeted power network is same or not!
## e.g., IEEE 57case in Pypower has different "rate A", "rate B", "rate C" values compared to PGLib.

ng = ppc['gen'].shape[0] # number of generators.
nbus = ppc['bus'].shape[0] # number of buses.
nl = ppc['branch'].shape[0] # total number of branches and transformers.

In [14]:
from scipy.stats.qmc import LatinHypercube

train_config = {
    'probType': 'acopf',
    'useCompl': True, # boolean type: whether to use completion (DC3, DeepLDE 기술)

    # GNN model parameters
    'n_gnn_layers': 3,
    'nfeature_dim': 2, # input dim
    'efeature_dim': 4, # edge feature dim 
    'hidden_dim': 40,
    'dropout_rate': 0.1,
    'K': 10, # 6 # only for TAGConv or ChebConv and GATConv (as multi-head)

    # DeepLDE hyperparameters
    'epochs': 30, # 10 (GPU RTX 4090), 8 (GPU A100)
    'batchSize': 5, # 6, (8) (GPU RTX 4090; GPU 다운 에러 발생..), 16 (GPU A100)
    'lr': 1e-4, # 1e-3
    'lr_w': 1e-4, # 1e-3
    'weight_decay': 1e-5,

    # LDF parameters
    'rho_init': 1e-4, # 0.01  
    's_init': 0.1,
    'p_iter_max': 10,
    'warmup_iter': 10, # 20, # 0 for non-warmup start cases
    'corrEps': 1e-4, # float type: correction procedure tolerance
}

eps_converge = train_config['corrEps']
valid_eps_converge = 1e-4
nepochs = train_config['epochs']
batch_size = train_config['batchSize']

train_loss_list = []
valid_loss_list = []
valid_eval_list = []

Kshot = 20 #
train_len_range = (0,Kshot) ## K-shot: 20, 10, 5, 1, 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(train_len_range) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

## Random sampling
# train_loader = torch_geometric.loader.DataLoader(random.sample(data.train_dataset, 100), batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

## Slicing
train_loader = torch_geometric.loader.DataLoader(data.train_dataset[train_len_range[0]:train_len_range[1]], batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

valid_loader = torch_geometric.loader.DataLoader(data.valid_dataset, batch_size=train_config['batchSize'], shuffle=False, drop_last=True)

# solver_net = GNNSolver(data, train_config)
solver_net = Edge_GNNSolver(data, train_config)

solver_net.to(DEVICE)

print(solver_net)

Edge_GNNSolver(
  (layers): ModuleList(
    (0): EdgeAggregation()
    (1): TransformerConv(120, 40, heads=10)
    (2): EdgeAggregation()
    (3): TransformerConv(120, 40, heads=10)
    (4): EdgeAggregation()
    (5): TransformerConv(120, 40, heads=10)
  )
  (flatten): Linear(in_features=26400, out_features=439, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [15]:
#################################### SETTING THE LOGS ####################################
result_path = os.path.join(ROOT_DIRECTORY, "results")
folder_name = "FT_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
              + str(train_config["rho_init"]) + "_rho_init_" + str(train_config["warmup_iter"]) + "_warmup_iter" 
# folder_name = "Local_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
#                + str(train_config["rho_init"]) + "_rho_init_" + str(train_config["warmup_iter"]) + "_warmup_iter"


log_path = os.path.join(result_path, folder_name, "logs")
data_tracking_path = os.path.join(result_path, folder_name, "data_tracking")

Path(log_path).mkdir(parents=True, exist_ok=True)
Path(data_tracking_path).mkdir(parents=True, exist_ok=True)
fileh = logging.FileHandler(os.path.join(log_path, "log.txt"), 'a')
global_logger.addHandler(fileh)

* Load pretrained GraphOPF model

In [16]:
pretrained_global_graphlde = torch.load("./models/global_model/checkpoint_Global.pth", weights_only=False)

* Transfer the weights from the pretrained GraphOPF

In [17]:
global_pretrained_state_dict = pretrained_global_graphlde
target_state_dict = solver_net.state_dict()

layer_names = list(target_state_dict.keys())
# load global model
for name in layer_names[:-2]:
    if name in target_state_dict.keys() and name in global_pretrained_state_dict.keys():
        # print(name)
        target_state_dict[name] = global_pretrained_state_dict[name].clone()

solver_net.load_state_dict(target_state_dict)

# # Freezing the half-GNN layers
# for i, (name, param) in enumerate(solver_net.named_parameters()):
#     if i <= 32: # 5, 10, 21
#         # print(name)
#         param.requires_grad = False
#     else:
#         print(name)
#         param.requires_grad = True

<All keys matched successfully>

* Training method: LD framework

In [18]:
stats = {}

# NOTE: LDF parameters.
LagM_sp_gen = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_gen = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_bus = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

warmup_iter = train_config["warmup_iter"] # the warmup period: the NN is trained with an additional inner iteration before the first outer iteration.
rho_init = train_config["rho_init"]
# s_init = train_config["s_init"]

rho = rho_init
# s = s_init

rho_iter = 0
s_iter = 0

p_iter_max = train_config["p_iter_max"]
p_iter_max_sum = p_iter_max

d = 0 # 0 for static case otherwise use 5"
beta = 0 # 0.001 # 0 for static case otherwise use 1

lr_w = train_config["lr_w"] # 이거 증가해도 되지 않을지?
lr = train_config["lr"]

print_interval = 1

train_start_time = time.time()
for i in range(nepochs):
    epoch_stats = {}

    ################### TRAINING PHASE ###################
    solver_net.train()
    if i<warmup_iter:
        ######### WARM-UP PERIOD #########
        if i == 0:
            solver_opt = optim.Adam(solver_net.parameters(), lr=lr_w) # this will be reinitalized after warmup stage
            print("Warmup start!")

        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            # start_time = time.time()
            solver_opt.zero_grad()

            # DNN + NR (equality constraint)
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    else:
        if i == warmup_iter:
            print("Warmup ended!")
            # data.ref_freedom = False

            # ineq_lag_flag = True # for the first warmup end epoch, consider lag. multipliers update of ineq.
        # elif (i - warmup_iter)%ineq_lag_flag_trigger == 0:
        #     # print("Doing test... continue this case..")
        #     print("consider lagrangian multipliers update for ineq. constraints!")
        #     ineq_lag_flag = True
        # else:
        #     ineq_lag_flag = False

        # for every (updated) p_iter_max_sum time.
        if (i - warmup_iter)%p_iter_max_sum == 0:
            ######### Outer Interation: calculate step size of lagrangian multipliers update #########
            if i> warmup_iter:
                print("current epoch %d || p_iter_max updated : %d -> %d" %(i, p_iter_max, p_iter_max + d))
                # s = s_init * (1/(1+beta*(s_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                # s_iter += 1
                # print("mu iter updated : %d -> %d" %(s_iter-1, s_iter))

                rho = rho_init * (1/(1+beta*(rho_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                rho_iter += 1
                print("rho iter updated : %d -> %d" %(rho_iter-1, rho_iter))
                p_iter_max = p_iter_max + d
                p_iter_max_sum += p_iter_max

            with torch.no_grad():
                print("Lambda updated at %d epoch" %i)
                solver_net.eval()
                for Xtrain in train_loader:
                    Xtrain = Xtrain.to(DEVICE)
                    #solver_opt.zero_grad()
                    Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                    # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

                    # LagM += rho*ineq_violation(data, Xtrain.x, Yhat_train) # Update the lagrangian multiplier.
                    LagM_sp_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[:2] # Update the lagrangian multiplier.
                    LagM_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2:2+2*ng] # Update the lagrangian multiplier.
                    LagM_bus += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng:2+2*ng+2*nbus] # Update the lagrangian multiplier.
                    LagM_line += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng+2*nbus:] # Update the lagrangian multiplier.

            solver_opt = optim.Adam(solver_net.parameters(), lr = lr)

        solver_net.train()
        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            solver_opt.zero_grad()
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, obj_train, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    if (i == 0) or (i%print_interval == 0):
        print(
            'Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))
    global_logger.info('Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))
    
    train_loss_list.append(np.mean(epoch_stats['train_loss']))
    # valid_eval_list.append(np.mean(epoch_stats['valid_eval']))
train_end_time = time.time()
total_train_time = train_end_time - train_start_time

Warmup start!
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It m

Epoch 0: train loss 3740.0767, train obj 57.8793, ineq max 735.6464, ineq mean 0.3320, ineq p_g num viol 1.0000, ineq q_g num viol 31.5500, ineq v_m num viol 0.0000, ineq line_theraml num viol 174.6000, eq max 0.0025, eq mean 0.0000


Epoch 0: train loss 3740.0767, train obj 57.8793, ineq max 735.6464, ineq mean 0.3320, ineq p_g num viol 1.0000, ineq q_g num viol 31.5500, ineq v_m num viol 0.0000, ineq line_theraml num viol 174.6000, eq max 0.0025, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performanc

Epoch 1: train loss 2013.9886, train obj 58.3377, ineq max 460.4421, ineq mean 0.1763, ineq p_g num viol 0.5500, ineq q_g num viol 23.9500, ineq v_m num viol 0.0000, ineq line_theraml num viol 124.7500, eq max 0.0018, eq mean 0.0000


Epoch 1: train loss 2013.9886, train obj 58.3377, ineq max 460.4421, ineq mean 0.1763, ineq p_g num viol 0.5500, ineq q_g num viol 23.9500, ineq v_m num viol 0.0000, ineq line_theraml num viol 124.7500, eq max 0.0018, eq mean 0.0000
outines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batche

...

Epoch 27: train loss 60.2140, train obj 56.6450, ineq max 0.6934, ineq mean 0.0003, ineq p_g num viol 0.0000, ineq q_g num viol 2.6500, ineq v_m num viol 0.0000, ineq line_theraml num viol 8.8000, eq max 0.0020, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   

Epoch 28: train loss 58.8490, train obj 56.2051, ineq max 0.4616, ineq mean 0.0002, ineq p_g num viol 0.0000, ineq q_g num viol 1.5000, ineq v_m num viol 0.0000, ineq line_theraml num viol 8.9500, eq max 0.0020, eq mean 0.0000


Epoch 28: train loss 58.8490, train obj 56.2051, ineq max 0.4616, ineq mean 0.0002, ineq p_g num viol 0.0000, ineq q_g num viol 1.5000, ineq v_m num viol 0.0000, ineq line_theraml num viol 8.9500, eq max 0.0020, eq mean 0.0000
nes are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines

Epoch 29: train loss 57.9774, train obj 55.7685, ineq max 0.4318, ineq mean 0.0002, ineq p_g num viol 0.0000, ineq q_g num viol 2.3500, ineq v_m num viol 0.0000, ineq line_theraml num viol 8.7500, eq max 0.0017, eq mean 0.0000


Epoch 29: train loss 57.9774, train obj 55.7685, ineq max 0.4318, ineq mean 0.0002, ineq p_g num viol 0.0000, ineq q_g num viol 2.3500, ineq v_m num viol 0.0000, ineq line_theraml num viol 8.7500, eq max 0.0017, eq mean 0.0000


In [19]:
total_train_time

95.41801691055298

In [20]:
# Save the training history

# train_loss_list
# epoch_stats['train_loss']
# (np.array(train_loss_list)).tolist()

global_logger.info("train_loss_list:{}".format((np.array(train_loss_list)).tolist()))

data_tracking = {
                "train_loss_list": (np.array(train_loss_list)).tolist(),
                }
with open(os.path.join(data_tracking_path, "metrics.pickle"), 'wb') as handle:
    pickle.dump(data_tracking, handle, protocol=pickle.HIGHEST_PROTOCOL)

train_loss_list:[3740.07666015625, 2013.9886474609375, 890.2147216796875, 341.9786071777344, 157.84103393554688, 131.17083740234375, 137.64266967773438, 139.44766235351562, 138.77191162109375, 137.37283325195312, 127.75528717041016, 104.57249450683594, 100.60921478271484, 91.26201629638672, 86.38346099853516, 83.87835693359375, 80.34751892089844, 77.68312072753906, 75.34810638427734, 73.0541000366211, 74.51127624511719, 71.37760162353516, 69.23513793945312, 67.3834457397461, 64.10466003417969, 62.9830322265625, 61.5407600402832, 60.213951110839844, 58.8489990234375, 57.977394104003906]


* Evaluation (using validation set)

In [21]:
len(data.test_dataset)

200

In [14]:
# load pretrained_model
# solver_net = torch.load(r'./models/pretrained_models/graphlde_2312_pretrained_model_20_real_slack_chebconv_final.pt', weights_only=False)
# solver_net = torch.load(r'./models/pretrained_models/graphlde_2312_pretrained_model_20_real_slack_chebconv_final_(weight_init_default).pt', weights_only=False)


In [22]:
from pypower.api import makeYbus
Ybus, Yf, Yt = makeYbus(data.baseMVA, data.ppc['bus'], data.ppc['branch'])
# branch thermal limit information
flow_max = (data.ppc['branch'][:, 5] / data.baseMVA)**2
flow_max[flow_max == 0] = np.inf
flow_max = torch.tensor(flow_max, dtype=torch.float32).to(data.device)

test_len = 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

test_loader = torch_geometric.loader.DataLoader(data.test_dataset[test_len:], batch_size=1, shuffle=False, drop_last=True)
# test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

solver_net.eval()
test_stats = {}
test_eps_converge = 1e-4

# LagM = torch.ones(1, 2*ng + 2*nbus + 2*nl).to(DEVICE) # shape: (1, num_inequalities)
LagM_sp_g = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_q_g = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_v_m = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line_l = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

solve_time = []
for (i, Xtest) in enumerate(test_loader):
    Xtest = Xtest.to(DEVICE)

    start_time = time.time()
    Y = solver_net(Xtest, n_means, n_stds, e_means, e_stds)
    end_time = time.time()

    solve_time += [end_time - start_time]

    ## line thermal limit
    pg, qg, vm, va = data.get_yvars(Y)
    vr = vm*torch.cos(va)
    vi = vm*torch.sin(va)
    vz = torch.complex(vr, vi) # complex voltage

    # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
    If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(data.device) @ vz.T
    It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(data.device) @ vz.T

    # Calculate the apparent power S
    Sf = vz[:,data.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
    St = vz[:,data.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
    Sff = Sf * torch.conj(Sf)
    Stt = St * torch.conj(St)

    # calculate the line thermal limit constraints violation
    diff_Sf = Sff.real - flow_max
    diff_St = Stt.real - flow_max
    # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

    line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
    line_limit_vio_St = torch.clamp(diff_St, 0)
    ###########################################

    # test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM)
    test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

    dict_agg(test_stats, 'time', end_time - start_time, op='sum')

    test_ineq_p_g = torch.cat([pg - data.pmax, data.pmin - pg], dim=1)
    test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(data.device)
    test_ineq_q_g = test_ineq_dist[:,2:2+2*ng]
    test_ineq_v_m = test_ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
    test_ineq_line_l = test_ineq_dist[:,2+2*ng+2*nbus:]

    dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
    # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

    dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

    pg_rate_torch = (((pg <= data.pmax) & (pg >= data.pmin)).sum()/ng)*100
    qg_rate_torch = (((qg <= data.qmax) & (qg >= data.qmin)).sum()/ng)*100
    dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

    v_rate_torch = (((vm <= data.vmax) & (vm >= data.vmin)).sum()/nbus)*100
    dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

    sff_rate_torch = ((Sff.real <= flow_max).sum()/nl)*100        
    stt_rate_torch = ((Stt.real <= flow_max).sum()/nl)*100        
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

    test_eq_real = test_eq_resid[:,:nbus]
    test_eq_react = test_eq_resid[:,nbus:]
    dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:nbus] <= 1e-2) & (test_eq_resid[:,:nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:nbus].shape[1]*100).detach().cpu().numpy())
    dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,nbus:] <= 1e-2) & (test_eq_resid[:,nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,nbus:].shape[1]*100).detach().cpu().numpy())

    print('Test batch {}: test loss {:.4f}, test obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}, p_g satisfication rate {:.4f}, q_g satisfication rate {:.4f}, v_m satisfication rate {:.4f}, test_line_limit_satisfication_rate {:.4f}, test_line_limit_satisfication_rate_Sf {:.4f}, test_line_limit_satisfication_rate_St {:.4f}, active eq satisfication rate {:.4f}, reactive eq satisfication rate {:.4f}'.format(
                i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_obj_cost']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
                np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']),
                np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean']), np.mean(test_stats['test_p_g_satisfication rate (%)']), np.mean(test_stats['test_q_g_satisfication rate (%)']), np.mean(test_stats['test_v_m_satisfication rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']), np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_active_eq_satisfication rate (%)']), np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))

#     print('Test batch {}: test loss {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, line_limit satisfication rate Sf {:.4f}, line_limit satisfication rate St {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
#             i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
#             np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']),
#             np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean'])))


Test batch 0: test loss 57.1048, test obj 55.4614, ineq max 0.3158, ineq mean 0.0001, ineq q_g num viol 1.0000, ineq v_m num viol 0.0000, eq max 0.0026, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 99.5454, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 99.8672, test_line_limit_satisfication_rate_Sf 99.8672, test_line_limit_satisfication_rate_St 99.8672, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 1: test loss 57.1423, test obj 55.4646, ineq max 0.3146, ineq mean 0.0002, ineq q_g num viol 2.0000, ineq v_m num viol 0.0000, eq max 0.0023, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 99.0909, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 99.8672, test_line_limit_satisfication_rate_Sf 99.8672, test_line_limit_satisfication_rate_St 99.8672, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 2: test loss 57.1840,

* Arithmetic mean

In [23]:
## Calculate the results of GraphLDE

print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

print("\n")
print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

print("\n")
print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
# print("DeepLDE time (ms) <== average value for test dataset:", (test_stats['time']/1000)*1e3)
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)

# 
global_logger.info('GraphLDE obj. value for test samples: {}'.format(round(np.mean(test_stats['test_obj_cost'])*10000, 4)))
global_logger.info('GraphLDE eq. mean for test samples: {}'.format(np.mean(test_stats['test_eq_mean'])))
global_logger.info('GraphLDE eq. max for test samples: {}'.format(np.mean(test_stats['test_eq_max'])))
global_logger.info('GraphLDE eq. active mean for test samples: {}'.format(np.mean(test_stats['test_eq_real_mean'])))
global_logger.info('GraphLDE eq. active max for test samples: {}'.format(np.mean(test_stats['test_eq_real_max'])))
global_logger.info('GraphLDE eq. reactive mean for test samples: {}'.format(np.mean(test_stats['test_eq_react_mean'])))
global_logger.info('GraphLDE eq. reactive max for test samples: {}'.format(np.mean(test_stats['test_eq_react_max'])))
global_logger.info('\n GraphLDE ineq. mean for test samples: {}'.format(np.mean(test_stats['test_ineq_mean'])))
global_logger.info('GraphLDE ineq. max for test samples: {}'.format(np.mean(test_stats['test_ineq_max'])))
global_logger.info('GraphLDE ineq. p_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_mean'])))
global_logger.info('GraphLDE ineq. p_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_max'])))
global_logger.info('GraphLDE ineq. q_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_mean'])))
global_logger.info('GraphLDE ineq. q_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_max'])))
global_logger.info('GraphLDE ineq. v_m mean for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_mean'])))
global_logger.info('GraphLDE ineq. v_m max for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_max'])))
global_logger.info('GraphLDE ineq. line_l mean for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_mean'])))
global_logger.info('GraphLDE ineq. line_l max for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_max'])))
global_logger.info('\n GraphLDE p_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_p_g_satisfication rate (%)'])))
global_logger.info('GraphLDE q_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_q_g_satisfication rate (%)'])))
global_logger.info('GraphLDE v_m satisfication rate for test samples: {}'.format(np.mean(test_stats['test_v_m_satisfication rate (%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_Sf for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_St for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_St(%)'])))
global_logger.info('GraphLDE active eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_active_eq_satisfication rate (%)'])))
global_logger.info('GraphLDE reactive eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))

GraphLDE obj. value for test samples: 554859.25
GraphLDE eq. mean for test samples: 4.24352110712789e-05
GraphLDE eq. max for test samples: 0.0018528338987380266
GraphLDE eq. active mean for test samples: 2.7150455935043283e-05
GraphLDE eq. active max for test samples: 0.0008663550252094865
GraphLDE eq. reactive mean for test samples: 5.7719964388525113e-05
GraphLDE eq. reactive max for test samples: 0.0018527810461819172

 GraphLDE ineq. mean for test samples: 0.0001570727617945522
GraphLDE ineq. max for test samples: 0.3624127209186554
GraphLDE ineq. p_g mean for test samples: 0.0
GraphLDE ineq. p_g max for test samples: 0.0


GraphLDE obj. value for test samples:  554859.25
GraphLDE eq. mean for test samples:  4.243521e-05
GraphLDE eq. max for test samples:  0.0018528339
GraphLDE eq. active mean for test samples:  2.7150456e-05
GraphLDE eq. active max for test samples:  0.000866355
GraphLDE eq. reactive mean for test samples:  5.7719964e-05
GraphLDE eq. reactive max for test samples:  0.001852781


GraphLDE ineq. mean for test samples:  0.00015707276
GraphLDE ineq. max for test samples:  0.36241272
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  0.00014081363
GraphLDE ineq. q_g max for test samples:  0.04426266
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0002788406
GraphLDE ineq. line_l max for test samples:  0.36241272


GraphLDE p_g satisfication rate for test samples:  99.99999
GraphLDE q_g satisfication rate for test samp

GraphLDE ineq. q_g mean for test samples: 0.00014081363042350858
GraphLDE ineq. q_g max for test samples: 0.044262658804655075
GraphLDE ineq. v_m mean for test samples: 0.0
GraphLDE ineq. v_m max for test samples: 0.0
GraphLDE ineq. line_l mean for test samples: 0.0002788405981846154
GraphLDE ineq. line_l max for test samples: 0.3624127209186554

 GraphLDE p_g satisfication rate for test samples: 99.99999237060547
GraphLDE q_g satisfication rate for test samples: 99.08181762695312
GraphLDE v_m satisfication rate for test samples: 100.0
GraphLDE test_line_limit_satisfication_rate_Sf for test samples: 99.85414123535156
GraphLDE test_line_limit_satisfication_rate_St for test samples: 99.85396575927734
GraphLDE active eq satisfication rate for test samples: 100.0
GraphLDE reactive eq satisfication rate for test samples: 100.0


* Harmonic mean

In [13]:
import statistics as st

## Calculate optimality gap
print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", st.harmonic_mean(test_stats['test_eq_mean'])) # print("LDF eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", st.harmonic_mean(test_stats['test_eq_max'])) # print("LDF eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", st.harmonic_mean(test_stats['test_eq_real_mean'])) # print("LDF eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", st.harmonic_mean(test_stats['test_eq_real_max'])) # print("LDF eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", st.harmonic_mean(test_stats['test_eq_react_mean'])) # print("LDF eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", st.harmonic_mean(test_stats['test_eq_react_max'])) # print("LDF eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))
print("\n")
print("GraphLDE ineq. mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_mean'])) # print("LDF ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", st.harmonic_mean(test_stats['test_ineq_max'])) # print("LDF ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_mean'])) # print("LDF ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_max'])) # print("LDF ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_mean'])) # print("LDF ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_max'])) # print("LDF ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_mean'])) # print("LDF ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_max'])) # print("LDF ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_mean'])) # print("LDF ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_max'])) # print("LDF ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))
print("\n")

print("GraphLDE p_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_p_g_satisfication rate (%)'].reshape(-1))) # print("LDF p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_q_g_satisfication rate (%)'].reshape(-1))) # print("LDF q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_v_m_satisfication rate (%)'].reshape(-1))) # print("LDF v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_St(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_active_eq_satisfication rate (%)'].reshape(-1))) # print("LDF active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_reactive_eq_satisfication rate (%)'].reshape(-1))) # print("LDF reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)


GraphLDE obj. value for test samples:  610214.4623
GraphLDE eq. mean for test samples:  3.956575960567648e-05
GraphLDE eq. max for test samples:  0.0021566857409128872
GraphLDE eq. active mean for test samples:  1.1870647493645656e-05
GraphLDE eq. active max for test samples:  0.0004547926142803058
GraphLDE eq. reactive mean for test samples:  6.71491826127369e-05
GraphLDE eq. reactive max for test samples:  0.0021566857409128872


GraphLDE ineq. mean for test samples:  0.010152037373214202
GraphLDE ineq. max for test samples:  4.90801989010103
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  0.029213124778608622
GraphLDE ineq. q_g max for test samples:  3.1647619408894716
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0164928368762264
GraphLDE ineq. line_l max for test samples:  4.889800643915613


GraphLDE

/home/super/anaconda3/envs/skjdeep/lib/python3.10/statistics.py:428: RuntimeWarning: divide by zero encountered in divide
  T, total, count = _sum(w / x if w else 0 for w, x in zip(weights, data))


In [10]:
# optimality gap
################### UNIFORM DISTRIBUTION - RANDOM SAMPLING ###################
## +/-20% perturbation <== Training time: 11min 13.9s
# MATPOWER: cost - 452260  // solve time - 3213.30  (ms)
# GraphLDE: cost - 459897.7661 // solve time - 68.06545734405518 (ms)

abs((459897.7661-452260)/452260)*100

1.6887998275328369